<a href="https://colab.research.google.com/github/jstrend/math/blob/main/%EA%B0%80%EC%9A%B0%EC%8A%A4_%EC%A1%B0%EB%8D%98_%EC%86%8C%EA%B1%B0%EB%B2%95p296.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sympy as sp


# --------------------------------------------------
# 1. 연립방정식 입력
# --------------------------------------------------

def input_linear_system():
    m = int(input("방정식의 개수를 입력하세요: "))
    n = int(input("미지수의 개수를 입력하세요: "))

    print(f"\n{m}×{n} 계수행렬 A를 입력하세요.")
    print("각 행의 계수를 공백으로 구분하여 입력하세요.")
    print("정수, 소수, 분수(예: 1/2)를 입력할 수 있습니다.\n")

    matrix_data = []

    for i in range(m):
        while True:
            values = input(f"계수행렬 {i + 1}행: ").split()

            if len(values) != n:
                print(f"계수를 정확히 {n}개 입력하세요.")
                continue

            try:
                row = [sp.Rational(value) for value in values]
                matrix_data.append(row)
                break

            except ValueError:
                print("올바른 숫자 형식으로 다시 입력하세요.")

    print("\n상수항 벡터 b를 입력하세요.")

    while True:
        values = input(f"상수항 {m}개: ").split()

        if len(values) != m:
            print(f"상수항을 정확히 {m}개 입력하세요.")
            continue

        try:
            constant_data = [sp.Rational(value) for value in values]
            break

        except ValueError:
            print("올바른 숫자 형식으로 다시 입력하세요.")

    A = sp.Matrix(matrix_data)
    b = sp.Matrix(constant_data)

    return A, b


# --------------------------------------------------
# 2. 행렬 출력
# --------------------------------------------------

def print_step(matrix, step, description):
    print(f"\n단계 {step}: {description}")
    sp.pprint(matrix)


# --------------------------------------------------
# 3. 가우스-조던 소거법
# --------------------------------------------------

def gauss_jordan_elimination(A, b):
    """
    확대행렬 [A|b]를 기약행 사다리꼴로 변환한다.
    """

    augmented = A.row_join(b)

    rows = A.rows
    cols = A.cols

    pivot_row = 0
    pivot_columns = []
    step = 0

    print("\n초기 확대행렬 [A | b]")
    sp.pprint(augmented)

    # 계수행렬의 각 열에 대해 피벗 탐색
    for pivot_col in range(cols):

        # 현재 피벗 열에서 0이 아닌 원소 탐색
        selected_row = None

        for row in range(pivot_row, rows):
            if augmented[row, pivot_col] != 0:
                selected_row = row
                break

        # 현재 열에 피벗이 없으면 다음 열로 이동
        if selected_row is None:
            continue

        # 행 교환
        if selected_row != pivot_row:
            augmented.row_swap(selected_row, pivot_row)

            step += 1
            print_step(
                augmented,
                step,
                f"R{pivot_row + 1} ↔ R{selected_row + 1}"
            )

        # 피벗을 1로 만들기
        pivot_value = augmented[pivot_row, pivot_col]

        if pivot_value != 1:
            augmented.row_op(
                pivot_row,
                lambda value, j: sp.simplify(value / pivot_value)
            )

            step += 1
            print_step(
                augmented,
                step,
                f"R{pivot_row + 1} ← "
                f"(1/{pivot_value})R{pivot_row + 1}"
            )

        # 피벗 열의 나머지 원소를 모두 0으로 만들기
        for row in range(rows):

            if row == pivot_row:
                continue

            factor = augmented[row, pivot_col]

            if factor != 0:
                augmented.row_op(
                    row,
                    lambda value, j: sp.simplify(
                        value - factor * augmented[pivot_row, j]
                    )
                )

                step += 1
                print_step(
                    augmented,
                    step,
                    f"R{row + 1} ← R{row + 1} "
                    f"- ({factor})R{pivot_row + 1}"
                )

        pivot_columns.append(pivot_col)
        pivot_row += 1

        # 모든 행에 피벗을 만들었으면 종료
        if pivot_row == rows:
            break

    return augmented, pivot_columns


# --------------------------------------------------
# 4. 해의 종류 판정 및 출력
# --------------------------------------------------

def analyze_solution(rref_matrix, pivot_columns, number_of_unknowns):
    rows = rref_matrix.rows
    n = number_of_unknowns

    print("\n" + "=" * 50)
    print("최종 기약행 사다리꼴 행렬")
    print("=" * 50)
    sp.pprint(rref_matrix)

    # 모순행 검사: [0 0 ... 0 | c], c ≠ 0
    for row in range(rows):
        coefficients_zero = all(
            rref_matrix[row, col] == 0 for col in range(n)
        )

        if coefficients_zero and rref_matrix[row, n] != 0:
            print("\n모순행이 존재합니다.")

            print(
                f"0 = {rref_matrix[row, n]}이므로 "
                "이 연립방정식은 해가 없습니다."
            )

            return "no_solution", None

    rank = len(pivot_columns)

    # 유일한 해
    if rank == n:
        solutions = [sp.Integer(0)] * n

        for row, pivot_col in enumerate(pivot_columns):
            solutions[pivot_col] = rref_matrix[row, n]

        print("\n유일한 해가 존재합니다.")

        for i, solution in enumerate(solutions):
            print(f"x{i + 1} = {solution}")

        return "unique", sp.Matrix(solutions)

    # 무수히 많은 해
    free_columns = [
        col for col in range(n)
        if col not in pivot_columns
    ]

    parameters = sp.symbols(f"t1:{len(free_columns) + 1}")
    solutions = [sp.Integer(0)] * n

    # 자유변수 설정
    for col, parameter in zip(free_columns, parameters):
        solutions[col] = parameter

    # 기본변수를 자유변수로 표현
    for row, pivot_col in enumerate(pivot_columns):
        expression = rref_matrix[row, n]

        for col, parameter in zip(free_columns, parameters):
            expression -= rref_matrix[row, col] * parameter

        solutions[pivot_col] = sp.simplify(expression)

    print("\n무수히 많은 해가 존재합니다.")
    print("자유변수를 매개변수로 나타내면 다음과 같습니다.\n")

    for i, solution in enumerate(solutions):
        sp.pprint(sp.Eq(sp.Symbol(f"x{i + 1}"), solution))

    return "infinite", sp.Matrix(solutions)


# --------------------------------------------------
# 5. 프로그램 실행
# --------------------------------------------------

A, b = input_linear_system()

print("\n① 계수행렬 A")
sp.pprint(A)

print("\n② 상수항 벡터 b")
sp.pprint(b)

rref_matrix, pivot_columns = gauss_jordan_elimination(A, b)

solution_type, solution = analyze_solution(
    rref_matrix,
    pivot_columns,
    A.cols
)

# 유일한 해인 경우 검산
if solution_type == "unique":
    print("\n③ 검산: A × x")

    verification = sp.simplify(A * solution)
    sp.pprint(verification)

    print("\n상수항 벡터 b")
    sp.pprint(b)

    if verification == b:
        print("\n검산 결과: A × x = b이므로 계산된 해가 정확합니다.")
    else:
        print("\n검산 결과를 다시 확인해야 합니다.")